# PTQ (INT8)

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


In [ ]:
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
else :
  save_dir = '../files/save/'

## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 1s 0us/step


In [ ]:
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

## Load Baseline Model for MNIST

In [ ]:
model = keras.models.load_model(save_dir + 'baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9904000163078308


## LiteRT 모델로 변환 (PTQ (INT8))

In [ ]:
def representative_data_gen():
  for input_value in tf.data.Dataset.from_tensor_slices(train_images).batch(1).take(100):
    yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
converter.representative_dataset = representative_data_gen

tflite_model = converter.convert()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
tflite_ptq_int8_file = save_dir + 'mnist_ptq_int8.tflite'
open(tflite_ptq_int8_file, 'wb').write(tflite_model)

66416

## File size 비교 : Baseline model vs PQT (INT8) model

In [ ]:
import os
tflite_baseline_model_file = save_dir + 'mnist_baseline_model.tflite'
print("Size of Baseline LiteRT Model file : {}".format(os.path.getsize(tflite_baseline_model_file)))
print("Size of PTQ(INT8) LiteRT Model file : {}".format(os.path.getsize(tflite_ptq_int8_file)))

Size of Baseline LiteRT Model file : 233596
Size of PTQ(INT8) LiteRT Model file : 66416


## LiteRT 설치 및 Interpreter 로딩

In [ ]:
!pip install ai-edge-litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 77.8 MB/s eta 0:00:00


In [ ]:
from ai_edge_litert.interpreter import Interpreter

## interpreter 생성 (Baseline model)

In [ ]:
interpreter_base = Interpreter(model_path=str(tflite_baseline_model_file))
interpreter_base.allocate_tensors()

## interpreter 생성 (PTQ (INT8))

In [ ]:
interpreter_ptq_int8 = Interpreter(model_path=str(tflite_ptq_int8_file))
interpreter_ptq_int8.allocate_tensors()

## input/output dtype 및 quantization factor 확인 (PTQ (INT8))

In [ ]:
input_dtype = interpreter_ptq_int8.get_input_details()[0]['dtype']
output_dtype = interpreter_ptq_int8.get_output_details()[0]['dtype']
input_scale, input_zero = interpreter_ptq_int8.get_input_details()[0]['quantization']
output_scale, output_zero = interpreter_ptq_int8.get_output_details()[0]['quantization']

print('input dtype: ', input_dtype)
print('output dtype: ', output_dtype)
print('input scale factor, zero point: {}, {}'.format(input_scale, input_zero))
print('output scale factor, zero point: {}, {}'.format(output_scale, output_zero))

input dtype:  <class 'numpy.int8'>
output dtype:  <class 'numpy.int8'>
input scale factor, zero point: 0.003921568859368563, -128
output scale factor, zero point: 0.00390625, -128


## 추론 실행 (PTQ (INT8))

In [ ]:
test_image = np.expand_dims(test_images[0], axis=0)

input_index = interpreter_ptq_int8.get_input_details()[0]["index"]
output_index = interpreter_ptq_int8.get_output_details()[0]["index"]

test_image = ((test_image / input_scale) + input_zero).astype(input_dtype)
interpreter_ptq_int8.set_tensor(input_index, test_image)

interpreter_ptq_int8.invoke()

output_tensor = interpreter_ptq_int8.get_tensor(output_index)
predictions = (output_tensor.astype(np.float32) - output_zero) * output_scale

print(output_tensor)
print(predictions)
print(np.argmax(predictions))
print(test_labels[0])

[[-128 -128 -128 -128 -128 -128 -128  127 -128 -128]]
[[0.         0.         0.         0.         0.         0.
  0.         0.99609375 0.         0.        ]]
7
7


## Test data 기반 accuracy 평가

In [ ]:
def evaluate_model(interpreter):
  input_details = interpreter.get_input_details()
  output_details = interpreter.get_output_details()

  input_index = input_details[0]["index"]
  output_index = output_details[0]["index"]

  prediction_digits = []
  for test_image in test_images:
    if input_details[0]['dtype'] == np.int8:
      input_scale, input_zero = input_details[0]["quantization"]
      test_image = (test_image / input_scale + input_zero).astype(input_details[0]['dtype'])

    test_image = np.expand_dims(test_image, axis=0)
    interpreter.set_tensor(input_index, test_image)

    interpreter.invoke()

    output = interpreter.get_tensor(output_index)
    if output_details[0]['dtype'] == np.int8:
      output = (output.astype(np.float32) - output_zero) * output_scale
    digit = np.argmax(output)
    prediction_digits.append(digit)

  accurate_count = 0
  for index in range(len(prediction_digits)):
    if prediction_digits[index] == test_labels[index]:
      accurate_count += 1
  accuracy = accurate_count * 1.0 / len(prediction_digits)

  return accuracy

In [ ]:
print(evaluate_model(interpreter_base))
print(evaluate_model(interpreter_ptq_int8))

0.9904
0.9903
